# Midden-laag spanning Stedin

In [2]:
from pathlib import Path
import geopandas as gpd
import folium 
from shapely.geometry import LineString, Polygon, box
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import shape

import numpy as np
import geopandas as gpd
from scipy.spatial import Voronoi
from shapely.geometry import Polygon

import rasterio
import geopandas as gpd
from shapely.geometry import Point
from rasterio.transform import Affine

In [3]:
import ra2ce.network.networks_utils as nut
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.network_config_data import (NetworkConfigData,NetworkSection,)
from ra2ce.network.network_wrappers.osm_network_wrapper.osm_network_wrapper import (OsmNetworkWrapper,)
from ra2ce.network.exporters.geodataframe_network_exporter import GeoDataFrameNetworkExporter
from ra2ce.network.exporters.multi_graph_network_exporter import MultiGraphNetworkExporter


In [4]:
root_dir = Path(r'C:\python\powerpath\data')
assert root_dir.exists()

In [5]:
static_path = root_dir.joinpath('static_OD')
network_path = static_path.joinpath('network')
hazard_path = static_path.joinpath("hazard")
output_path = static_path.joinpath("output_graph")

# RA2CE stuff

In [6]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import pickle
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
    OriginsDestinationsSection
)
from ra2ce.configuration.config_wrapper import ConfigWrapper
from ra2ce.analysis.analysis_config_wrapper import AnalysisConfigWrapper
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisConfigData
from ra2ce.ra2ce_handler import Ra2ceHandler
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisSectionBase, AnalysisSectionLosses, ProjectSection
from ra2ce.analysis.analysis_config_data.enums.analysis_damages_enum import AnalysisDamagesEnum
from ra2ce.analysis.analysis_config_data.enums.analysis_losses_enum import AnalysisLossesEnum
from ra2ce.analysis.analysis_config_data.enums.damage_curve_enum import DamageCurveEnum
from ra2ce.analysis.analysis_config_data.enums.event_type_enum import EventTypeEnum
from ra2ce.analysis.analysis_config_data.enums.weighing_enum import WeighingEnum
from ra2ce.analysis.losses.multi_link_origin_closest_destination import MultiLinkOriginClosestDestination
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_wrapper import NetworkConfigWrapper
from ra2ce.network.graph_files.graph_files_collection import GraphFilesCollection
from ra2ce.network.graph_files.graph_file import GraphFile
from ra2ce.network.graph_files.network_file import NetworkFile
from ra2ce.network.hazard.hazard_names import HazardNames

c:\python\ra2ce\ra2ce_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Functions

def get_all_files(directory: str) -> list[Path]:
    """
    Get all file names in the specified directory.

    Args:
        directory (str): The path to the directory.

    Returns:
        List[str]: A list of file names in the directory.
    """
    p = Path(directory)
    return [file for file in p.iterdir() if file.is_file()]

def read_pickle(file_path: str):
    """
    Read a pickle file.

    Args:
        file_path (str): The path to the pickle file.

    Returns:
        The object stored in the pickle file.
    """
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data

def read_gpkg_to_gdf(file_path: str, layer: str = None) -> gpd.GeoDataFrame:
    """
    Read a GeoPackage file into a GeoDataFrame.

    Args:
        file_path (str): The path to the GeoPackage file.
        layer (str, optional): The specific layer to read from the GeoPackage. If None, reads the default layer.

    Returns:
        gpd.GeoDataFrame: The GeoDataFrame created from the GeoPackage file.
    """
    # Read the geopackage file into a GeoDataFrame
    gdf = gpd.read_file(file_path, layer=layer)
    
    return gdf

In [8]:
Extent_path = network_path.joinpath("try_study_area.shp")
Extent = gpd.read_file(Extent_path, driver='ESRI Shapefile')
shapely_polygon = Extent.geometry.iloc[0]


In [9]:
# import rasterio
# import numpy as np
# from pathlib import Path

# import rasterio
# import numpy as np
# from pathlib import Path

# # Function to process each raster file
# def process_raster(hazard_file, output_dir):
#     with rasterio.open(hazard_file) as src:
#         # Read the data
#         data = src.read(1)
        
#         # Replace NoData values with 0
#         if src.nodata is not None:
#             data[data == src.nodata] = 0
#         # Set values below zero to 0
#         data[data < 0] = 0

#         # Update the metadata to reflect changes
#         meta = src.meta.copy()
#         meta.update(nodata=0)  # Set new NoData value to 0

#         # Write the modified data to a new file in the output directory
#         output_file = output_dir / f"{hazard_file.stem}_processed.tif"
#         with rasterio.open(output_file, 'w', **meta) as dst:
#             dst.write(data, 1)


# # Process each hazard file
# for hazard_file in hazard_files:
#     process_raster(hazard_file, hazard_path_processed)
#     print(f"Processed {hazard_file} to {hazard_path_processed}")

In [10]:
#First we define which roads we will want to download from OSM to create a network with
_network_section = NetworkSection(
    network_type=NetworkTypeEnum.DRIVE,
    polygon=Extent_path,
    save_gpkg=True,
    road_types=[RoadTypeEnum.MOTORWAY,RoadTypeEnum.MOTORWAY_LINK,RoadTypeEnum.PRIMARY, RoadTypeEnum.PRIMARY_LINK,RoadTypeEnum.TRUNK, RoadTypeEnum.SECONDARY,RoadTypeEnum.SECONDARY_LINK, RoadTypeEnum.TERTIARY, RoadTypeEnum.RESIDENTIAL], 
)

_network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    output_path=output_path,
    network=_network_section,
)

_graph,_gdf = OsmNetworkWrapper.get_network_from_polygon(_network_config_data,polygon=shapely_polygon)

c:\python\ra2ce\ra2ce_env\Lib\site-packages\osmnx\simplification.py:513: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  merged = convert.graph_to_gdfs(G, edges=False)["geometry"].buffer(tolerance).unary_union
c:\python\ra2ce\ra2ce_env\Lib\site-packages\osmnx\simplification.py:560: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = node_clusters.centroid
c:\python\ra2ce\ra2ce_env\Lib\site-packages\osmnx\simplification.py:513: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  merged = convert.graph_to_gdfs(G, edges=False)["geometry"].buffer(tolerance).unary_union
c:\python\r

In [11]:
# Export the graph
_exporter = MultiGraphNetworkExporter(basename='base_graph', export_types=['gpkg', 'pickle'])
_exporter.export(export_path=root_dir.joinpath('static_OD','output_graph'), export_data=_graph)

# Export the network
_exporter = GeoDataFrameNetworkExporter(basename='base_network', export_types=['gpkg', 'pickle'])
_exporter.export(export_path=root_dir.joinpath('static_OD','output_graph'), export_data=_gdf)

In [12]:
#polygon_file = network_path.joinpath("buffer_polygon_network.geojson")
#assert polygon_file.exists()

hazard_files = get_all_files(hazard_path)
hazard_crs = "EPSG:4326" # for the hackathon case => "EPSG:4326" 

for hazard_file in hazard_files:
    print (hazard_file)

C:\python\powerpath\data\static_OD\hazard\clipped_hazard_map.tif


In [13]:
path_hazard = Path(r'C:\python\powerpath\data\static_OD\hazard\clipped_hazard_map.tif')

In [16]:
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.network_config_data import OriginsDestinationsSection


_network_section = NetworkSection(
    source=SourceEnum.OSM_DOWNLOAD,
    network_type=NetworkTypeEnum.DRIVE,
    polygon=shapely_polygon,
    save_gpkg=True,
    road_types=[RoadTypeEnum.MOTORWAY,RoadTypeEnum.MOTORWAY_LINK,RoadTypeEnum.PRIMARY, RoadTypeEnum.PRIMARY_LINK,RoadTypeEnum.TRUNK, RoadTypeEnum.SECONDARY,RoadTypeEnum.SECONDARY_LINK, RoadTypeEnum.TERTIARY, RoadTypeEnum.RESIDENTIAL], 
)

_hazard = HazardSection(
    hazard_map=[path_hazard],   #[Path(geotiff_files[0])],
    hazard_field_name= ['waterdepth'],
    aggregate_wl = AggregateWlEnum.MAX,
    hazard_crs = "EPSG:4326",
    overlay_segmented_network=False,
)

_origin_destination_section = OriginsDestinationsSection(
    origins=network_path.joinpath("Manual_O.shp"),
    destinations=network_path.joinpath("Manual_D.shp"),
    origins_names="A",
    destinations_names="B",
    id_name_origin_destination="ID",
    origin_count="pop",
    origin_out_fraction=1,
    category="category",
)


_network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    output_path=output_path,
    network=_network_section,
    hazard= _hazard,
    origins_destinations=_origin_destination_section
)


handler = Ra2ceHandler.from_config(_network_config_data, None)
handler.configure()
handler.run_analysis()


In [24]:
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisSectionBase, AnalysisSectionLosses, ProjectSection
from ra2ce.analysis.analysis_config_data.enums.analysis_damages_enum import AnalysisDamagesEnum
from ra2ce.analysis.analysis_config_data.enums.analysis_losses_enum import AnalysisLossesEnum
from ra2ce.analysis.analysis_config_data.enums.damage_curve_enum import DamageCurveEnum
from ra2ce.analysis.analysis_config_data.enums.event_type_enum import EventTypeEnum
from ra2ce.analysis.analysis_config_data.enums.weighing_enum import WeighingEnum
from ra2ce.analysis.losses.multi_link_origin_closest_destination import MultiLinkOriginClosestDestination


_od_section = AnalysisSectionLosses(
    name='accessibility',
    analysis=AnalysisLossesEnum.OPTIMAL_ROUTE_ORIGIN_CLOSEST_DESTINATION,
    #aggregate_wl= AggregateWlEnum.MAX,
    threshold=0.2,
    weighing=WeighingEnum.LENGTH,
    calculate_route_without_disruption=True,
    save_gpkg=True,
    save_csv=True,
)

_analysis_config_data = AnalysisConfigData(
    project=ProjectSection(name='accessibility'),
    analyses=[_od_section],
    output_path=output_path,
    root_path=root_dir,
    static_path=static_path,
    
)

from ra2ce.analysis.analysis_config_wrapper import AnalysisConfigWrapper
from ra2ce.network.hazard.hazard_names import HazardNames

_analysis_config_wrapper = AnalysisConfigWrapper()
_analysis_config_wrapper.config_data = _analysis_config_data
_analysis_config_data.hazard_names = HazardNames.from_config(_analysis_config_wrapper)


In [25]:
# Run analysis
_handler = Ra2ceHandler.from_config(_network_config_data, _analysis_config_data)
_handler.configure()
_handler.run_analysis()

Finding optimal routes to repair: 100%|██████████| 396/396 [00:00<00:00, 12544.42it/s]


[AnalysisResultWrapper(results_collection=[AnalysisResult(analysis_result=<networkx.classes.multigraph.MultiGraph object at 0x0000018305A2D4D0>, analysis_config=AnalysisSectionLosses(name='accessibility', save_gpkg=True, save_csv=True, analysis=<AnalysisLossesEnum.OPTIMAL_ROUTE_ORIGIN_CLOSEST_DESTINATION: 5>, weighing=<WeighingEnum.LENGTH: 1>, production_loss_per_capita_per_hour=nan, traffic_period=<TrafficPeriodEnum.DAY: 1>, hours_per_traffic_period=0, trip_purposes=[<TripPurposeEnum.NONE: 0>], resilience_curves_file=None, traffic_intensities_file=None, values_of_time_file=None, threshold=0.2, threshold_destinations=nan, equity_weight='', calculate_route_without_disruption=True, buffer_meters=nan, category_field_name='', save_traffic=False, event_type=<EventTypeEnum.NONE: 0>, risk_calculation_mode=<RiskCalculationModeEnum.NONE: 0>, risk_calculation_year=0), output_path=WindowsPath('C:/python/powerpath/data/static_OD/output_graph'), _custom_name='accessibility_origins'), AnalysisResult